# uniko — the Pydantic IO layer

This is the same memory walkthrough as `uniko_memory_demo.ipynb`, but every value
crossing the FFI boundary is a typed, schema-bearing **Pydantic** model.

The layer (`uniko.models`) is a pure-Python overlay on the native extension:

- **Output models** — `ContextBundle`, `Answer`, `GoalView`, … built with
  `Model.from_native(native)`.
- **Input specs** — `GoalSpec`, `TurnSpec`, `IngestSourceSpec`, … validated *before*
  they cross into Rust, then turned into native builders with `spec.to_native()`.
- **Typed handles** — `TypedUniko` / `TypedAgent` / `TypedSession` auto-accept specs
  and auto-return models, so you never call `from_native` / `to_native` yourself.

`pydantic>=2` is now a runtime dependency of the `uniko` package.

> **Note.** The first `observe()` / `recall()` downloads the embedding/NER models
> (~850 MB) to `~/.uni_cache/`.

## 1. Open a typed instance & read its config

`TypedUniko` starts the journey typed from the top; `config()` returns a `UnikoConfigModel`.

In [ ]:
import uniko
from uniko.models import TypedUniko

uni = await TypedUniko.in_memory()
config = uni.config()             # -> UnikoConfigModel
print(type(config).__name__)
config.model_dump()

## 2. Input validation — catch mistakes *before* the FFI boundary

Specs use `extra="forbid"` and field constraints, so a bad payload raises a precise `ValidationError` in Python instead of an opaque error from Rust.

In [ ]:
from pydantic import ValidationError
from uniko.models import GoalSpec, IngestSourceSpec, LlmSpecAdapter

# Empty title is rejected.
try:
    GoalSpec(title="")
except ValidationError as e:
    print("empty title  ->", e.errors()[0]["msg"])

# A typo'd keyword is caught, not silently dropped.
try:
    GoalSpec(title="x", descriptoin="oops")
except ValidationError as e:
    print("typo kwarg   ->", e.errors()[0]["msg"], e.errors()[0]["loc"])

# IngestSource is a discriminated one-of: text / bytes / path.
try:
    IngestSourceSpec(origin={"source": "nope", "content": "x"})
except ValidationError as e:
    print("bad source   ->", e.errors()[0]["type"])

# LlmSpec discriminated union: openai_with_key_env requires key_env.
try:
    LlmSpecAdapter.validate_python({"provider": "openai_with_key_env", "alias": "a", "model_id": "m"})
except ValidationError as e:
    print("missing field ->", e.errors()[0]["loc"], e.errors()[0]["type"])

## 3. Observe via `TurnSpec` → `ObserveResult` model

`TypedSession.observe` accepts a spec (or a native `Turn`) and returns an `ObserveResult` model. A spec can carry metadata and nested attachments.

In [ ]:
from uniko.models import TurnSpec, IngestSourceSpec

agent = uni.agent("assistant")
session = agent.session("trip-planning")

turns = [
    TurnSpec(sender_id="alice", content="I love hiking in the Dolomites.", message_id="m1",
             metadata={"topic": "outdoors"}),
    TurnSpec(sender_id="alice", content="Here are my route notes.", message_id="m2",
             attachments=[IngestSourceSpec.from_text("Day 1: Tre Cime loop", id="notes-1")]),
]
for t in turns:
    result = await session.observe(t)        # -> ObserveResult
    print(result.message_node_id, "entities:", result.extracted_entities)
result.model_dump()

## 4. Recall → `ContextBundle` model (the FastAPI-ready artifact)

`recall` returns a `ContextBundle` model. `model_dump_json()` serializes it; `model_json_schema()` is exactly the schema FastAPI emits for a `response_model`.

In [ ]:
bundle = await agent.recall("hiking")        # -> ContextBundle
print("items:", len(bundle), "coverage:", round(bundle.coverage, 3))
for item in bundle.items[:3]:
    print(f"  [{item.kind}] {item.score:.3f}  {item.content[:60]!r}")
print()
print(bundle.model_dump_json()[:200], "...")

In [ ]:
import json
schema = bundle.model_json_schema()
print("top-level properties:", sorted(schema["properties"]))
print("nested $defs        :", sorted(schema.get("$defs", {})))

## 5. `Answer` — schema now, generation later

`agent.answer()` needs an LLM (`LlmSpec.openai(...)` on the builder), so it isn't run here. But the `Answer` model is fully defined — note `citations` is a real field (the native side exposes it as a method) so it appears in `model_dump()` and the schema.

In [ ]:
from uniko.models import Answer
print("Answer fields:", list(Answer.model_json_schema()["properties"]))

## 6. Data views as models

`agent.data.message(id)` / `.artifact(id)` return `MessageView` / `ArtifactView` models (or `None`).

In [ ]:
msg = await agent.data.message("m1")         # -> MessageView | None
print(type(msg).__name__, "->", msg.sender_id, msg.timestamp.isoformat())
art = await agent.data.artifact("notes-1")   # -> ArtifactView | None
print(type(art).__name__, "->", art.text[:40])

## 7. Goals & tasks via specs → views

`TypedGoals.create` takes a `GoalSpec`; reads return `GoalView` / `TaskView` / `GoalContext` models. Goals are owned by the agent's participant, which the earlier `observe` already registered… for a *different* sender — so we register `assistant` first.

In [ ]:
from uniko.models import GoalSpec, TaskSpec

await agent.session("setup").observe(TurnSpec(sender_id="assistant", content="kickoff"))
goals = agent.goals

await goals.create(GoalSpec(title="Ship the Pydantic IO layer", goal_id="g1",
                            metrics={"models": 13, "specs": 6}))
await goals.create_task(TaskSpec(title="write the notebook", goal_id="g1", task_id="t1", priority=0.9))
await goals.start("g1")

goal = await goals.get("g1")                 # -> GoalView
print(goal.title, "| phase:", goal.phase, "| metrics:", goal.metrics)
ctx = await goals.context("g1")              # -> GoalContext
print("tasks:", [t.title for t in ctx.tasks])

## 8. Logic surfaces

`abduce` is the one logic output we model (`AbductionResult`). Dynamic Cypher (`query`) and Locy (`run_rule`) results stay raw `list[dict]` — they have no fixed shape to model.

In [ ]:
from uniko.models import AbductionResult

await agent.define_rule("reachable", "CREATE RULE reachable AS MATCH (a:Episode) YIELD KEY a")
result = await agent.abduce("ABDUCE reachable WHERE a.kind = 'query'")   # -> AbductionResult
print("modifications:", len(result))

rows = await agent.query("MATCH (n:Message) RETURN n LIMIT 2")           # stays list[dict]
print("query rows are dicts:", all(isinstance(r, dict) for r in rows))

## 9. FastAPI integration

Because outputs are Pydantic models and input specs validate request bodies, the
SDK drops straight into FastAPI. The models *are* the OpenAPI schema:

```python
from fastapi import FastAPI
from uniko.models import ContextBundle, GoalSpec

app = FastAPI()

@app.post("/recall", response_model=ContextBundle)
async def recall(q: str):
    return ContextBundle.from_native(await agent.recall(q))

@app.post("/goals", response_model=int)
async def create_goal(spec: GoalSpec):          # request body validated by Pydantic
    return await agent.goals.create(spec.title, **spec.to_create_kwargs())
```

The cell below prints the JSON Schema FastAPI would publish for those endpoints —
no server required.

In [ ]:
from uniko.models import ContextBundle, GoalSpec
print("response schema (ContextBundle):", sorted(ContextBundle.model_json_schema()["properties"]))
print("request schema  (GoalSpec)     :", sorted(GoalSpec.model_json_schema()["properties"]))
print("GoalSpec rejects extras        :", GoalSpec.model_json_schema().get("additionalProperties"))

## Recap

- **Outputs** are typed models (`from_native`) with `model_dump()` / JSON Schema.
- **Inputs** are validated specs (`to_native`) — errors surface in Python, early.
- **Typed handles** (`TypedUniko` & friends) give specs-in / models-out with no
  manual conversion.
- The whole layer slots into **FastAPI** via `response_model` / request bodies.

Reach for raw native objects when you want zero overhead or the untyped surfaces
(dynamic Cypher rows, Locy params); reach for the models everywhere validation or
serialization matters.